# IT3091 — Telco Customer Churn | Member 4: Decision Tree (Corrected)

**Purpose:** Train, tune and interpret an interpretable Decision Tree churn classifier using **training data only**. This notebook revises Member 4's original Decision Tree work, retaining its balanced tree, parameter grid, model rules, feature importance and low/medium/high risk profiling.

**Critical correction:** The original notebook evaluated baseline and tuned models and profiled customers on the held-out test set prematurely. **This notebook never loads `X_test_raw.csv` or `y_test.csv`**. It uses stratified training-only cross-validation and nested out-of-fold predictions for provisional model comparison and risk profiling. Previously viewed test scores must not be treated as a pristine final holdout evaluation.

Run Notebook 2 first, then run this notebook **from `notebooks/`**, with scikit-learn, pandas, numpy, matplotlib and joblib installed. Results are generated, not hard-coded.

## 1. Imports and deterministic setup

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import (StratifiedKFold, GridSearchCV,
                                     cross_validate, cross_val_predict)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report)
from sklearn.base import clone

RANDOM_STATE = 42
REPORTS = Path('../reports')
MODELS = Path('../models')
REPORTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
print('Ready: training-only Decision Tree analysis')

## 2. Load the shared **training** split
Do not load the held-out test split for this member’s model development.

In [ ]:
X_train = pd.read_csv('../data/processed/X_train_raw.csv')
y_df = pd.read_csv('../data/processed/y_train.csv')
y_train = y_df['Churn'] if 'Churn' in y_df.columns else y_df.squeeze('columns')
y_train = pd.to_numeric(y_train, errors='raise').astype(int)
assert len(X_train) == len(y_train), 'X/y row mismatch'
assert set(y_train.unique()) == {0, 1}, 'Unexpected binary label values'
assert not X_train.isna().any().any(), 'Missing feature values: inspect Notebook 2'
print('Training X:', X_train.shape, 'Training y:', y_train.shape)
print('Class counts:\n', y_train.value_counts().sort_index())
print('Held-out test set: NOT LOADED')

## 3. Pipeline and baseline balanced Decision Tree
Trees do not need standard scaling. One-hot encoding is fitted inside each training fold to prevent preprocessing leakage.

In [ ]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
binary_features = ['SeniorCitizen']
required = numeric_features + binary_features
assert all(c in X_train.columns for c in required), 'Required feature missing'
categorical_features = [c for c in X_train.columns if c not in required]

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('binary', 'passthrough', binary_features)
])

baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE))
])
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'accuracy': 'accuracy', 'precision': 'precision',
           'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc',
           'average_precision': 'average_precision'}
print('Numerical:', numeric_features)
print('Categorical count:', len(categorical_features))
print('Binary:', binary_features)

## 4. Baseline cross-validation
These scores are **training-only 5-fold means**, not test-set results.

In [ ]:
base_cv = cross_validate(baseline, X_train, y_train, cv=outer_cv,
                         scoring=scoring, n_jobs=1)
baseline_summary = pd.DataFrame([{
    'model': 'Balanced Decision Tree baseline',
    **{m: float(np.mean(base_cv['test_' + m])) for m in scoring},
    **{m + '_std': float(np.std(base_cv['test_' + m])) for m in scoring}
}])
display(baseline_summary.round(4))
baseline_summary.to_csv(REPORTS / 'decision_tree_baseline_cv_results.csv', index=False)

## 5. Hyperparameter tuning on training data
Keep the **original Member 4 grid**: `max_depth` = 3, 5, 7, 10, None; `min_samples_leaf` = 5, 10, 20, 50. Fit an inner search on the whole training set to choose the **deployable training-only model**. Its `best_score_` is a selection score, **not** the unbiased performance estimate.

In [ ]:
params = {
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_leaf': [5, 10, 20, 50]
}
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(baseline, params, scoring='f1', cv=inner_cv,
                      n_jobs=1, refit=True, return_train_score=True)
search.fit(X_train, y_train)
print('Selected training-only parameters:', search.best_params_)
print('Inner CV best F1 (selection-biased):', round(search.best_score_, 4))
pd.DataFrame(search.cv_results_).to_csv(REPORTS / 'decision_tree_tuning_results.csv', index=False)

## 6. Nested cross-validation for tuned Decision Tree
Each outer fold runs its **own inner grid search** using only that outer fold’s training portion. The outer predictions are therefore out-of-fold for the entire hyperparameter-selection procedure. Runtime: 5 outer folds × 3 inner folds × 20 parameter settings.

In [ ]:
nested_search = GridSearchCV(
    clone(baseline), params, scoring='f1', cv=inner_cv,
    n_jobs=1, refit=True
)
oof_prob = cross_val_predict(nested_search, X_train, y_train,
                             cv=outer_cv, method='predict_proba', n_jobs=1)[:, 1]

def evaluate_probabilities(y_true, prob, threshold=0.5):
    pred = (np.asarray(prob) >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, prob),
        'average_precision': average_precision_score(y_true, prob)
    }

nested_metrics = pd.DataFrame([{
    'model': 'Nested-tuned balanced Decision Tree',
    **evaluate_probabilities(y_train, oof_prob, 0.5)
}])
display(nested_metrics.round(4))
print('OOF confusion matrix (threshold 0.50):\n',
      confusion_matrix(y_train, oof_prob >= 0.5))
print('OOF report:\n', classification_report(y_train, oof_prob >= 0.5, zero_division=0))
nested_metrics.to_csv(REPORTS / 'decision_tree_nested_cv_results.csv', index=False)

## 7. Exploratory threshold sensitivity
These probabilities are generated by nested cross-validation, but **the threshold is chosen on the same out-of-fold predictions used to report the F1 score**. The winning threshold is **provisional**; its selected F1 is optimistic. The held-out test set remains unused in this notebook.

In [ ]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
threshold_df = pd.DataFrame([
    evaluate_probabilities(y_train, oof_prob, t) for t in thresholds
])
best_row = threshold_df.loc[threshold_df['f1'].idxmax()]
provisional_threshold = float(best_row['threshold'])
display(threshold_df.round(4))
print('Provisional threshold (best exploratory F1):', provisional_threshold)
threshold_df.to_csv(REPORTS / 'decision_tree_threshold_results.csv', index=False)

## 8. Fit final **training-only** model, extract readable rules
The model is fitted on all training records only, using hyperparameters selected by the inner search. Do **not** report its in-sample training score as generalization performance.

In [ ]:
final_tree = clone(search.best_estimator_)
final_tree.fit(X_train, y_train)
clf = final_tree.named_steps['classifier']
feature_names = final_tree.named_steps['preprocessor'].get_feature_names_out()
rules = export_text(clf, feature_names=list(feature_names))
(REPORTS / 'decision_tree_rules.txt').write_text(rules, encoding='utf-8')
print('Saved full rules to reports/decision_tree_rules.txt')
print('Preview (first 2500 characters):\n', rules[:2500])

## 9. Feature importance
Impurity-based importance is a relative description of this tree, **not causal evidence**. Importance can be distributed across correlated predictors and one-hot categories.

In [ ]:
importance_df = (pd.DataFrame({'Feature': feature_names,
                               'Importance': clf.feature_importances_})
                 .sort_values('Importance', ascending=False).reset_index(drop=True))
display(importance_df.head(15))
importance_df.to_csv(REPORTS / 'decision_tree_feature_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 6))
top = importance_df.head(15).iloc[::-1]
ax.barh(top['Feature'], top['Importance'])
ax.set_xlabel('Decision Tree impurity-based importance')
ax.set_title('Top 15 Decision Tree features — fitted on training data')
fig.tight_layout()
fig.savefig(REPORTS / 'decision_tree_feature_importance.png', dpi=160, bbox_inches='tight')
plt.show()

## 10. Training-only customer churn risk profiles
Use **out-of-fold training predictions**, not the prematurely examined test set. Risk bands (low < 0.30; medium 0.30–0.60; high ≥ 0.60) are illustrative, not independently validated business policies.

In [ ]:
risk_profile = X_train.copy()
risk_profile['Actual_Churn'] = y_train.to_numpy()
risk_profile['Predicted_Churn_Risk_OOF'] = oof_prob
risk_profile['Risk_Level'] = pd.cut(
    oof_prob, bins=[-0.001, 0.30, 0.60, 1.001],
    labels=['Low', 'Medium', 'High'], include_lowest=True
)
risk_summary = (risk_profile.groupby('Risk_Level', observed=False)
                .agg(customers=('Actual_Churn', 'size'),
                     actual_churn_rate=('Actual_Churn', 'mean'),
                     mean_predicted_risk=('Predicted_Churn_Risk_OOF', 'mean'))
                .reset_index())
display(risk_summary)
risk_profile.to_csv(REPORTS / 'decision_tree_risk_profiles_training_oof.csv', index=False)
risk_summary.to_csv(REPORTS / 'decision_tree_risk_summary_training_oof.csv', index=False)
print('Profiles are cross-validated TRAINING records, not future customer outcomes.')

## 11. Save model, provisional configuration and verify files

In [ ]:
model_path = MODELS / 'decision_tree.joblib'
joblib.dump(final_tree, model_path)
config = {
    'model': 'Decision Tree',
    'hyperparameters': search.best_params_,
    'provisional_threshold': provisional_threshold,
    'threshold_status': 'exploratory, selected on nested OOF predictions',
    'fit_data': 'shared training split only',
    'test_set_status': 'not read by this corrected notebook; original notebook previously accessed test'
}
(MODELS / 'decision_tree_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
loaded_tree = joblib.load(model_path)
assert np.allclose(loaded_tree.predict_proba(X_train.head(5)), final_tree.predict_proba(X_train.head(5)))
expected_files = [
    'decision_tree_baseline_cv_results.csv',
    'decision_tree_tuning_results.csv',
    'decision_tree_nested_cv_results.csv',
    'decision_tree_threshold_results.csv',
    'decision_tree_rules.txt',
    'decision_tree_feature_importance.csv',
    'decision_tree_feature_importance.png',
    'decision_tree_risk_profiles_training_oof.csv',
    'decision_tree_risk_summary_training_oof.csv'
]
for name in expected_files:
    assert (REPORTS / name).exists(), f'Missing: {name}'
print('Model reload matches original:', model_path)
print('Reports created:', len(expected_files))
print('No held-out test files read.')

## 12. Decision log and methodological limitations

| Decision | Choice and justification |
|---|---|
| Project goal | Binary churn classification, with explainable risk groups as a secondary objective |
| Shared split | Reuse Member 1’s training-only split; do not create a new train/test split |
| Baseline | Balanced Decision Tree, `random_state=42`, complete preprocessing `Pipeline` |
| Hyperparameters | Preserve Member 4's `max_depth` and `min_samples_leaf` search grid |
| Validation | 5-fold baseline CV; 5-fold outer / 3-fold inner **nested CV** for tuned out-of-fold predictions |
| Threshold | Compare several cutoffs; chosen cutoff is exploratory until independently assessed |
| Interpretation | Export tree rules, impurity importance and out-of-fold training risk groups |
| Test contamination caveat | The original Member 4 notebook previously viewed held-out test metrics. Do not call that test set fully untouched; coordinate final assessment with the group |
| Group comparison | Use training-only, suitably comparable validation estimates for LR, RF, XGBoost and DT; don't claim a winner using previously viewed test scores |

### Viva talking points

**Why Decision Tree?** It learns readable branching rules and nonlinear interactions. **Why tune depth and leaf size?** To reduce overfitting. **Why nested CV?** Because selecting hyperparameters and evaluating them on the same folds overstates generalization. **Why not use the test set now?** The test set should be reserved for final group evaluation, although the original version accessed it prematurely. **Why out-of-fold risk profiles?** To give a less biased illustration than predictions on the same records used to fit the tree. **Do feature importances imply causation?** No.

**Remaining group work:** compare all four models under a consistent validation protocol; settle a threshold and investigate calibration/business costs; run final assessment with transparent disclosure of the earlier test-set access.